In [1]:
from kaggle_secrets import UserSecretsClient
secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN")

In [2]:
from huggingface_hub import login
login(token=hf_token)

In [3]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="liuhaotian/LLaVA-CC3M-Pretrain-595K",
    filename="images.zip",
    repo_type="dataset",
    local_dir="./data"
)

images.zip:   0%|          | 0.00/6.46G [00:00<?, ?B/s]

'data/images.zip'

In [4]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="liuhaotian/LLaVA-CC3M-Pretrain-595K",
    filename="chat.json",
    repo_type="dataset",
    local_dir="/kaggle/working/data"
)

'/kaggle/working/data/chat.json'

In [ ]:
import zipfile

with zipfile.ZipFile('./data/images.zip', 'r') as zip_ref:
    zip_ref.extractall('./data/images')

In [ ]:
import os
os.remove('./data/images.zip')

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import json
from torch.utils.data import DataLoader
from transformers import CLIPProcessor
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPVisionModel, AutoTokenizer, AutoModelForCausalLM

In [ ]:
from transformers import CLIPProcessor

class LLaVADataset(Dataset):
    def __init__(self, chat_json_path, image_folder, processor):
        with open(chat_json_path, 'r') as f:
            self.data = json.load(f)
        self.image_folder = image_folder
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = Image.open(f"{self.image_folder}/{sample['image']}").convert('RGB')
        image_tensor = self.processor(images=image, return_tensors='pt')['pixel_values'].squeeze(0)
        human = sample['conversations'][0]['value']
        gpt = sample['conversations'][1]['value']
        return image_tensor, human, gpt

In [ ]:
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
dataset = LLaVADataset('/kaggle/working/data/chat.json', '/kaggle/working/data/images', processor)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)
batch = next(iter(dataloader))
images, humans, gpts = batch
print(images.shape)

In [ ]:
class LLaVA(nn.Module):
    def __init__(self, vision_model, llm, tokenizer):
        super().__init__()
        self.vision_model = vision_model  # already on its device
        self.llm = llm                    # already on its device
        self.tokenizer = tokenizer
        self.W_proj = nn.Linear(1024, 2048, bias=False).to(next(llm.parameters()).device)
        
        for param in self.vision_model.parameters():
            param.requires_grad = False
        for param in self.llm.parameters():
            param.requires_grad = False

    def forward(self, images, humans, gpts):
        llm_device = next(self.llm.parameters()).device
        
        # CLIP on its device
        vision_outputs = self.vision_model(
            pixel_values=images.to(next(self.vision_model.parameters()).device),
            output_hidden_states=True
        )
        patch_features = vision_outputs.hidden_states[-2][:, 1:, :]
        
        # move to LLM device, project
        visual_tokens = self.W_proj(patch_features.to(llm_device))

        human_tokens = self.tokenizer(list(humans), return_tensors='pt', padding=True).to(llm_device)
        gpt_tokens = self.tokenizer(list(gpts), return_tensors='pt', padding=True).to(llm_device)

        human_embeddings = self.llm.model.embed_tokens(human_tokens['input_ids'])
        gpt_embeddings = self.llm.model.embed_tokens(gpt_tokens['input_ids'])

        combined = torch.cat([visual_tokens, human_embeddings, gpt_embeddings], dim=1)
        outputs = self.llm(inputs_embeds=combined.to(self.llm.dtype))
        logits = outputs.logits

        N = gpt_tokens['input_ids'].shape[1]
        answer_logits = logits[:, -(N+1):-1, :]
        answer_labels = gpt_tokens['input_ids']

        loss = F.cross_entropy(
            answer_logits.reshape(-1, 32000),
            answer_labels.reshape(-1)
        )
        return loss

In [ ]:
vision_model = CLIPVisionModel.from_pretrained("openai/clip-vit-large-patch14").to('cuda:0')
llm = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").to('cuda:1')
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

model = LLaVA(vision_model, llm, tokenizer)
optimizer = torch.optim.Adam(model.W_proj.parameters(), lr=2e-3)

In [ ]:
dataset = LLaVADataset(
    chat_json_path='/kaggle/working/data/chat.json',
    image_folder='/kaggle/working/data/images',
    processor=processor
)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)

In [ ]:
for epoch in range(1):
    for i, (images, humans, gpts) in enumerate(dataloader):
        
        optimizer.zero_grad()
        loss = model(images, humans, gpts)
        loss.backward()
        optimizer.step()
        
        if i % 10 == 0:
            print(f"Step {i} Loss: {loss.item():.4f}")
        
        if i % 1000 == 0:
            torch.save(model.W_proj.state_dict(), f'/kaggle/working/W_proj_step_{i}.pt')

torch.save(model.W_proj.state_dict(), '/kaggle/working/W_proj_final.pt')

In [ ]:
import torch
print(torch.cuda.memory_allocated(0) / 1e9, "GB on cuda:0")
print(torch.cuda.memory_allocated(1) / 1e9, "GB on cuda:1")